# 01 — EDA & Data Preparation

**Proje:** Credit Risk Scoring (Home Credit Default Risk)

Bu defter, projenin ilk durağı. Burada yalnızca ana eğitim tablosunu (`application_train.csv`) güvenli ve **bellek dostu** şekilde yükleyip ilk sağlık kontrollerini yapıyoruz. Ağır birleştirme (bureau, previous_application vb.) ve özellik mühendisliği sonraki defterlerde gelecek.

**Bu defterin hedefleri**
1. `data/raw/application_train.csv` dosyasını yükle.
2. Sayısal sütunları downcast ederek RAM kullanımını düşür (`float64 → float32`, küçük integer tipleri).
3. Hızlı sağlık kontrolü: `shape`, `dtypes`, hedef değişken (`TARGET`) dağılımı, eksik veri özeti.

In [ ]:
import sys
from pathlib import Path

print("Python executable:", sys.executable)
print("Python version   :", sys.version.split()[0])

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils import (
    load_csv,
    missing_value_report,
    split_feature_types,
    plot_target_distribution,
    plot_categorical_default_rate,
    plot_numeric_by_target,
    categorize_missing,
    plot_missing_top,
    missing_target_relationship,
    RAW_DIR,
)

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)
sns.set_theme(style="whitegrid", context="notebook")

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data dir: {RAW_DIR}")

## 1) `application_train.csv` yükleme

`src.utils.load_csv` fonksiyonu dosyayı `data/raw/` altından okur ve `reduce_mem_usage` ile sayısal tipleri downcast eder. Bu sayede aynı içerik çok daha az RAM ile taşınabilir hale gelir.

In [ ]:
app_train = load_csv("application_train.csv")
app_train.head()

## 2) Hızlı sağlık kontrolü

Aşağıdaki üç hücre veriyi tanımak için kritik:
- **Boyut & dtypes:** beklenen sütun sayısı ve indirgenmiş tipler
- **TARGET dağılımı:** sınıf dengesizliğinin (yaklaşık %8) doğrulanması
- **Eksik veri özeti:** ileride doldurma stratejisini şekillendirecek

In [ ]:
print(f"Shape: {app_train.shape}")
dtype_counts = app_train.dtypes.astype(str).value_counts()
print("\nDtype counts:")
print(dtype_counts)

In [ ]:
target_counts = app_train["TARGET"].value_counts().sort_index()
target_ratio = app_train["TARGET"].value_counts(normalize=True).sort_index()

target_summary = pd.DataFrame({"count": target_counts, "ratio": target_ratio})
target_summary.index.name = "TARGET"
target_summary

In [ ]:
missing_value_report(app_train, top=15)

## 3) `TARGET` dağılımı (görsel)

Sayısal tabloyu üst hücrede gördük. Aynı bilgiyi sınıf dengesizliğini somutlaştırmak için bar grafikle de gösterelim. Pozitif (temerrüt) sınıf payı yaklaşık `%8` — bu oran modelleme aşamasında `class_weight`, `scale_pos_weight` veya kalibrasyon stratejilerini doğrudan etkileyecek.

In [ ]:
overall_default_rate = float(app_train["TARGET"].mean())
print(f"Overall default rate: {overall_default_rate:.4f}")

fig, ax = plt.subplots(figsize=(6, 4))
plot_target_distribution(app_train, target="TARGET", ax=ax)
plt.tight_layout()
plt.show()

## 4) Özellik tipleri ayrımı

Modelleme öncesi sütunları üç gruba ayırıyoruz:
- **numeric:** doğrudan model girişi olabilir
- **categorical:** ileride encode edilmesi gereken (≤50 unique)
- **high_cardinality:** stratejisi farklı olacak (target encoding vb.)

`SK_ID_CURR` (kimlik) ve `TARGET` ayrımdan dışlanır.

In [ ]:
feature_groups = split_feature_types(app_train, exclude=["SK_ID_CURR", "TARGET"])

for group, cols in feature_groups.items():
    print(f"{group:>17}: {len(cols)}")

print("\nCategorical columns:")
print(feature_groups["categorical"])

## 5) Kategorik özellikler — TARGET'a göre temerrüt oranı

Aşağıda 4 önemli kategorik değişkenin her bir kategorisindeki **temerrüt oranı** (P(TARGET=1)) gösteriliyor. Kesik çizgi genel ortalama (`%8.07`).

Anlamlı sinyaller:
- `NAME_CONTRACT_TYPE`: nakit krediler genelde daha yüksek temerrütlü
- `CODE_GENDER`: erkek başvurucularda oran biraz daha yüksek
- `NAME_EDUCATION_TYPE`: eğitim seviyesi düştükçe risk artıyor
- `NAME_FAMILY_STATUS`: medeni hal segmentasyonu

In [ ]:
cat_focus = [
    "NAME_CONTRACT_TYPE",
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS",
]

fig, axes = plt.subplots(2, 2, figsize=(14, 9))
for ax, col in zip(axes.flat, cat_focus):
    plot_categorical_default_rate(
        app_train,
        col,
        target="TARGET",
        overall_rate=overall_default_rate,
        ax=ax,
    )
plt.tight_layout()
plt.show()

## 6) Kritik numerik özellikler — TARGET'a göre dağılım

`EXT_SOURCE_1/2/3` Home Credit yarışmasının en güçlü sayısal sinyalleri (dış kredi skor benzeri, 0–1 aralığında). `DAYS_BIRTH` ise negatif gün cinsinden olduğu için `AGE_YEARS = -DAYS_BIRTH / 365.25` dönüşümünü yapıyoruz. Eğri ayrılığı ne kadar belirginse, değişken o kadar bilgi verici.

In [ ]:
app_train["AGE_YEARS"] = (-app_train["DAYS_BIRTH"] / 365.25).astype("float32")

num_focus = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3", "AGE_YEARS"]

fig, axes = plt.subplots(2, 2, figsize=(13, 9))
for ax, col in zip(axes.flat, num_focus):
    plot_numeric_by_target(app_train, col, target="TARGET", ax=ax)
plt.tight_layout()
plt.show()

## 7) Eksik veri analizi — derinlik

Bölüm 2'de yalnızca en eksik 15 sütunu listelemiştik. Burada üç açıdan derinleşiyoruz:

1. **Kategorize et:** sütunları eksiklik yüzdesine göre kovalara ayır → ileride hangi sütunu *drop* edeceğimizi, hangisini *impute* edeceğimizi netleştir.
2. **Görselleştir:** en yüksek 25 sütun yatay bar grafikle.
3. **Eksiklik bir sinyal mi?** Bazı sütunlarda **eksik olmak** başlı başına temerrüt riskiyle ilişkili olabilir. Eksik vs. dolu satırlardaki `TARGET` ortalamasını karşılaştırıp **lift** hesaplayalım.

In [ ]:
missing_buckets = categorize_missing(
    app_train,
    low=0.2,
    mid=0.5,
    exclude=["SK_ID_CURR", "TARGET"],
)

bucket_summary = pd.DataFrame(
    {"column_count": {k: len(v) for k, v in missing_buckets.items()}}
)
bucket_summary.index.name = "bucket"

print("Missing-ratio buckets (excluding ID & TARGET):")
print(bucket_summary)
print(f"\nDrop adayları (>=70% eksik): {len(missing_buckets['extreme'])} sütun")
print(f"Impute adayları (20%-70%)  : {len(missing_buckets['medium']) + len(missing_buckets['high'])} sütun")
print(f"Hafif eksik (<=20%)        : {len(missing_buckets['low'])} sütun")
print(f"Hiç eksik yok              : {len(missing_buckets['none'])} sütun")

### 7.1) En çok eksiği olan 25 sütun

In [ ]:
fig, ax = plt.subplots(figsize=(9, 8))
plot_missing_top(app_train.drop(columns=["SK_ID_CURR", "TARGET"]), top=25, ax=ax)
plt.tight_layout()
plt.show()

### 7.2) Eksiklik = sinyal mi?

`lift = target_rate_missing / target_rate_not_missing`

- `lift ≈ 1`  : eksik olmak fark yaratmıyor → güvenle impute edilebilir.
- `lift > 1`  : eksiklerde temerrüt oranı daha yüksek → "is_missing" türetilmiş bayrak değişkeni faydalı olabilir.
- `lift < 1`  : eksik olanlar daha güvenli profili temsil ediyor olabilir (ör. ek belge gerektirmemiş müşteriler).

In [ ]:
missing_signal = missing_target_relationship(
    app_train,
    target="TARGET",
    min_missing_count=500,
    top=15,
)
print(f"Overall TARGET rate: {missing_signal.attrs['overall_target_rate']:.4f}\n")
missing_signal.style.format(
    {
        "missing_count": "{:,.0f}",
        "target_rate_missing": "{:.3%}",
        "target_rate_not_missing": "{:.3%}",
        "lift": "{:.3f}",
    }
)

### 7.3) Bulgular

- 122 sütundan büyük bir bölümü `BUILDING_*` ve `OWN_CAR_*` aileleridir; bunlarda eksiklik oranı tutarlı şekilde ~%50–70 civarında ve birlikte eksik oluyorlar (yani aynı başvuranlar bu blokları boş bırakmış).
- **`extreme` (≥%70)** kovasındaki sütunlar drop adayıdır — kalan ~%30 örnek üzerinden anlamlı bilgi çıkarmak zor.
- **`high/medium`** kovasındakiler için stratejimiz: median/mode ile impute + `is_missing_<col>` bayrak feature'ı (özellikle `lift` 1'den uzak olanlarda).
- **`EXT_SOURCE_1`** orta düzeyde eksik (~%56) ama hem değer hem eksiklik **güçlü TARGET sinyali** taşıyor — bu yüzden hem impute edilecek hem de bayrak değişkeni türetilecek.

Sonraki defterde (02) bu kararları feature mühendisliği aşamasında uygulayacağız.